# Tutorial 005 - Connecting Devices

In this tutorial we are going to create two special devices, one that has air going out and one that has air going in, then connecting an instances together.

First, go through the same steps as before importing and creating a namespace:

In [1]:
from bob.core import Device, Property, bind_model_namespace, dump
_namespace = bind_model_namespace("ex", "http://example/")

Rather than going through the design of a series of classes, we'll take advantage of some built into Bob (and print out the _method resolution order_ for the curious):

In [2]:
from bob.connections.air import AirInletConnectionPoint, AirOutletConnectionPoint
print(AirInletConnectionPoint.__mro__)

(<class 'bob.connections.air.AirInletConnectionPoint'>, <class 'bob.connections.air.AirConnectionPoint'>, <class 'bob.core.InletConnectionPoint'>, <class 'bob.core.ConnectionPoint'>, <class 'bob.core.Node'>, <class 'object'>)


Create some classes and instances:

In [3]:
class OutputDevice(Device):
    aocp: AirOutletConnectionPoint

class InputDevice(Device):
    aicp: AirInletConnectionPoint

out_device = OutputDevice(label="out_device")
in_device = InputDevice(label="in_device")

Now we can investigate the graph of the things we created:

In [4]:
dump()

@prefix bob: <http://data.ashrae.org/standard223/si-builder#> .
@prefix ex: <http://example/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
ex:00001 a s223:Device,
        ex:OutputDevice ;
    rdfs:label "out_device" ;
    s223:hasConnectionPoint ex:00002 ;
    ex:aocp ex:00002 .
ex:00002 a s223:OutletConnectionPoint,
        bob:AirConnectionPoint ;
    rdfs:label "out_device.aocp" ;
    s223:hasDirection s223:Direction-Outlet ;
    s223:hasMedium s223:Medium-Air ;
    s223:isConnectionPointOf ex:00001 .
ex:00003 a s223:Device,
        ex:InputDevice ;
    rdfs:label "in_device" ;
    s223:hasConnectionPoint ex:00004 ;
    ex:aicp ex:00004 .
ex:00004 a s223:InletConnectionPoint,
        bob:AirConnectionPoint ;
    rdfs:label "in_device.aicp" ;
    s223:hasDirection s223:Direction-Inlet ;
    s223:hasMedium s223:Medium-Air ;
    s223:isConnectionPointOf ex:00003 .


Notice that the fact that an output device has an outlet connection point is so interesting to Bob that it automatically created an instance of that class and bound it to the instance of the output device:

In [5]:
print(f"{out_device.aocp=}")

out_device.aocp=<AirOutletConnectionPoint out_device.aocp at http://example/00002>


So far these two devices aren't connected, but there is a simple expression to do that by overloading the shift-right operator:

In [6]:
out_device >> in_device

<InputDevice in_device at http://example/00003>

And the resulting graph, complete with an instance of a Connection with the appropriate medium:

In [7]:
dump()

@prefix bob: <http://data.ashrae.org/standard223/si-builder#> .
@prefix ex: <http://example/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix s223: <http://data.ashrae.org/standard223#> .
ex:00001 a s223:Device,
        ex:OutputDevice ;
    rdfs:label "out_device" ;
    s223:connectedThrough ex:00005 ;
    s223:connectedTo ex:00003 ;
    s223:hasConnectionPoint ex:00002 ;
    ex:aocp ex:00002 .
ex:00002 a s223:OutletConnectionPoint,
        bob:AirConnectionPoint ;
    rdfs:label "out_device.aocp" ;
    s223:connectsThrough ex:00005 ;
    s223:hasDirection s223:Direction-Outlet ;
    s223:hasMedium s223:Medium-Air ;
    s223:isConnectionPointOf ex:00001 .
ex:00003 a s223:Device,
        ex:InputDevice ;
    rdfs:label "in_device" ;
    s223:connectedFrom ex:00001 ;
    s223:connectedThrough ex:00005 ;
    s223:hasConnectionPoint ex:00004 ;
    ex:aicp ex:00004 .
ex:00004 a s223:InletConnectionPoint,
        bob:AirConnectionPoint ;
    rdfs:label "in_device.aicp" ;
 